In [7]:
#Importação e leitura de dados:

import pandas as pd
import numpy as np

df_vendas = pd.read_csv("C:\Projetos python\Analise de dados\Analise e Limpeza de Dados\datasets\origem\dirty_cafe_sales.csv",encoding='utf-8')

def extracao_dados():
    try:
        if df_vendas is not None and not df_vendas.empty:
            print('Extraindo dados...')
            return df_vendas
    except Exception as e:
        print(f'Não foi possivel extrair os dados: {e}')

df_analise = extracao_dados()


Extraindo dados...


In [9]:
# Etapa de tratamento dos dados:

# -tratando coluna de itens:
def tratamento(df_analise):
    try:
        if df_analise is not None and not df_analise.empty:
            lista_erros = ['NaN','ERROR','UNKNOWN']

            df_analise["Item"]= df_analise["Item"].str.strip().fillna("Não especificado")
            df_analise["Item"]=df_analise["Item"].replace(lista_erros,'Não especificado')


            # -tratando a quantidade:

            df_analise["Quantity"] = pd.to_numeric(df_analise["Quantity"], errors='coerce').fillna(0)
            df_analise["Total Spent"] = pd.to_numeric(df_analise["Total Spent"], errors='coerce').fillna(0)
            df_analise["Price Per Unit"] = pd.to_numeric(df_analise["Price Per Unit"], errors='coerce').fillna(0)

            filtro = (df_analise["Quantity"] == 0) & (df_analise["Price Per Unit"] > 0) & (df_analise["Total Spent"] > 0)

            df_analise.loc[filtro, "Quantity"] = df_analise["Total Spent"] / df_analise["Price Per Unit"]

            df_analise["Quantity"] = df_analise["Quantity"].astype(int)


            # -tratando o preco unitario:
            filtro_precoun = (df_analise["Price Per Unit"]==0) & (df_analise["Quantity"]>0) & (df_analise["Total Spent"]> 0)
            df_analise.loc[filtro_precoun,"Price Per Unit"] = df_analise["Total Spent"]/df_analise["Quantity"]

            df_analise["Price Per Unit"] = df_analise["Price Per Unit"].astype(float)

            # -tratando o preco total:

            filtro_total = (df_analise["Total Spent"]==0) & (df_analise["Quantity"]>0) & (df_analise["Price Per Unit"]>0)
            df_analise.loc[filtro_total,"Total Spent"] = df_analise["Price Per Unit"] * df_analise["Quantity"]

            df_analise["Total Spent"] = df_analise["Total Spent"].astype(float)

            # -tratando forma de pagamento:
            df_analise["Payment Method"] = df_analise["Payment Method"].fillna("Sem registro")
            df_analise["Payment Method"] = df_analise["Payment Method"].replace(lista_erros,"Sem registro")
            

            # -tratando localização:
            df_analise["Location"] = df_analise["Location"].replace(lista_erros,"Não especificado").fillna("In-store")

            # -tratando data
            df_analise = df_analise.dropna(subset=["Transaction Date"])
            df_analise = df_analise[df_analise["Transaction Date"] != 'Sem registro']
            df_analise["Transaction Date"] = pd.to_datetime(df_analise["Transaction Date"],errors='coerce')

            # -duplicatas
            df_analise = df_analise.drop_duplicates(subset=["Transaction ID"],keep='first')
            print('Tratando dados...')
            
        return df_analise
        
    except Exception as e:
        return print(f'Tratamento nao concluido: {e}')
    
df_analise = tratamento(df_analise)



    





Tratando dados...


In [10]:
  
def carga_dados():
    try:
        if df_analise is not None and not df_analise.empty:
            csv_path = 'C:\Projetos python\Analise de dados\Analise e Limpeza de Dados\datasets\origem\processados\coffee_sales.csv'
            df_analise.to_csv(csv_path,index=False)
            resultado = print('Arquivo carregado com sucesso!!!')
            return resultado
    except Exception as e:
        erro = print(f'Não foi possivel carregar a base tratada, erro: {e}')
        return erro
    
    
carga_dados()


Arquivo carregado com sucesso!!!


In [11]:
#Analise dos dados

#faturamento por itens vendidos na cafeteria:


print('Faturamento total por item vendido:\n')

fat_por_item = df_analise[["Item","Total Spent"]].groupby("Item").sum().sort_values(by=["Total Spent"],ascending=False)
display(fat_por_item)

quant_por_item = df_analise[["Item","Quantity"]].groupby("Item").sum().sort_values(by=["Quantity"],ascending=False)
display(quant_por_item)

md_preco_item = df_analise[["Item","Price Per Unit"]].groupby("Item").mean().sort_values(by=["Price Per Unit"],ascending=False)
display(md_preco_item)




Faturamento total por item vendido:



,Total Spent
Item,
Salad,16550.0
Sandwich,13016.0
Smoothie,12760.0
Juice,10017.0
Cake,9825.0
Não especificado,8091.5
Coffee,6804.0
Tea,4656.0
Cookie,3067.0


,Quantity
Item,
Coffee,3405
Juice,3341
Salad,3314
Cake,3272
Sandwich,3262
Smoothie,3188
Tea,3099
Cookie,3074
Não especificado,2766


,Price Per Unit
Item,
Salad,4.990901
Smoothie,3.984733
Sandwich,3.981395
Juice,2.997331
Cake,2.988909
Não especificado,2.896980
Coffee,1.996438
Tea,1.491237
Cookie,0.995169
